# Full Analysis — RQ1–RQ4 (4 LLMs, 30 methods × 2 conditions)
Models: **gpt-5.5-instant, gemini(-2.5-flash), llama(-3.3-70B Groq), deepseek(-v4-flash)**.
p-value + effect size + kết luận. Nguồn số liệu: `results/stats/unified.csv` (sinh bởi `compute_metric.py`).
Chạy: cần `pandas scipy scikit-posthocs` (xem `scripts/requirements-analysis.txt`).

In [ ]:
import pandas as pd, numpy as np
from itertools import combinations
from scipy import stats
import scikit_posthocs as sp

df = pd.read_csv('stats/unified.csv')   # 240 rows: model, bug, condition, line_cov, branch_cov, smells, ...
MODELS = ['gpt-5.5-instant','gemini','llama','deepseek']
print(len(df), 'rows'); df.head()

In [ ]:
def cliffs_delta(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    gt=(a[:,None]>b[None,:]).sum(); lt=(a[:,None]<b[None,:]).sum()
    d=(gt-lt)/(len(a)*len(b)); ad=abs(d)
    mag='negligible' if ad<0.147 else 'small' if ad<0.33 else 'medium' if ad<0.474 else 'large'
    return d,mag

# RQ1-RQ3: Kruskal-Wallis + Dunn(Bonferroni) + Cliff's delta
for metric,label in [('line_cov','RQ1 Line'),('branch_cov','RQ2 Branch'),('smells','RQ3 Smells')]:
    groups=[df[df.model==m][metric].values for m in MODELS]
    H,p=stats.kruskal(*groups)
    print(f'\n{label}: Kruskal-Wallis H={H:.3f}, p={p:.4g}', '=> significant' if p<0.05 else '=> ns')
    if p<0.05:
        dunn=sp.posthoc_dunn(df,val_col=metric,group_col='model',p_adjust='bonferroni')
        for a,b in combinations(MODELS,2):
            if dunn.loc[a,b]<0.05:
                d,mag=cliffs_delta(df[df.model==a][metric],df[df.model==b][metric])
                print(f'   {a} vs {b}: Dunn p={dunn.loc[a,b]:.4g}, Cliff d={d:+.3f} ({mag})')

In [ ]:
# RQ4: few-shot (C2) vs zero-shot (C1), Wilcoxon signed-rank (paired by bug)
for metric,label in [('line_cov','Line'),('branch_cov','Branch'),('smells','Smells')]:
    piv=df.pivot_table(index=['model','bug'],columns='condition',values=metric).dropna(subset=['C1','C2'])
    c1,c2=piv['C1'].values,piv['C2'].values
    if np.allclose(c1,c2):
        print(f'RQ4 {label}: identical'); continue
    W,p=stats.wilcoxon(c1,c2)
    print(f'RQ4 {label} (pooled): W={W:.1f}, p={p:.4g}', '**' if p<0.05 else '')

## Kết luận (số liệu thật)
- **RQ1 Line Coverage:** khác biệt có ý nghĩa (H=17.07, p=6.8e-4). Chỉ **gpt-5.5 > deepseek** qua Bonferroni (Cliff δ=0.420, medium).
- **RQ2 Branch Coverage:** cùng dạng (H=16.31, p=9.8e-4); gpt-5.5 > deepseek (δ=0.401, medium).
- **RQ3 Test Smell Density:** khác biệt rất mạnh (H=57.44, p=2.1e-12); gemini ~0 smell, gpt-5.5 & llama nhiều smell nhất (δ tới 0.483, large).
- **RQ4 Few-shot vs Zero-shot:** few-shot **giảm smell** có ý nghĩa (pooled p=1.3e-9; gpt p=2.5e-6, llama p=8.2e-6) nhưng **không cải thiện coverage** (line p=0.25, branch p=0.99).
- Xếp hạng tổng thể (compile/detect/coverage): gpt-5.5-instant > gemini ≈ llama > deepseek.